# Module 4B: Non-Price Pre_Regulation_Ext Staging

This notebook creates the non-price staging tables from `Pre_Regulation_Ext.dtsx` in `main.regtech_ops_stg`.

**Replicates SSIS package:** `Pre_Regulation_Ext.dtsx` (truncate/reload)

**Targets created:**
- `bi_output_regtechops_reg_ext_trade_getinstrument` — Instrument reference
- `bi_output_regtechops_reg_ext_trade_instrumentmetadata` — Instrument metadata
- `bi_output_regtechops_reg_ext_dictionarycurrency` — Currency dictionary
- `bi_output_regtechops_reg_ext_dictionarycurrencytype` — Currency type dictionary
- `bi_output_regtechops_reg_ext_hedgeexecutionlog` — Hedge execution log (date-windowed)
- `bi_output_regtechops_reg_ext_hedgehbcexecutionlog` — HBC execution log
- `bi_output_regtechops_reg_ext_hedgehbcorderlog` — HBC order log
- `bi_output_regtechops_reg_instruments_ext` — Instruments SCD snapshot (from PRODUCTION `gold_regreportdb_prod_dbo_reg_instruments_scd`)

**Source change (2026-06-11):** `reg_instruments_ext` now uses `main.regtech.gold_regreportdb_prod_dbo_reg_instruments_scd` (exact SP source mirror). This table provides `IsMifid`, `IsMifidByFCA`, and `IsFuture` natively — no LEFT JOIN to `bronze_etoro_trade_futuresmetadata` needed. Previous source (`gold_regtech_reg_instruments_scd`) had `IsMifidByESMA` (wrong column), cleared IsMifid flags (only 316 tagged vs 10,526 in production), and no `IsFuture` column.

**Note:** `Reg_Ext_CustomerLatinName` is excluded (source table not found). `Reg_Ext_MigrationInOut_STG` depends on Module 5.

In [0]:
dbutils.widgets.text("report_date", "2026-06-09", "Report Date (YYYY-MM-DD)")

report_date = dbutils.widgets.get("report_date")
print(f"Running Module 4B: Non-Price Staging for report_date = {report_date}")

In [0]:
%sql
-- Reg_Ext_Trade_GetInstrument: full snapshot of Trade.GetInstrument
-- SSIS parity: full table snapshot (no date filter in Pre_Regulation_Ext.dtsx)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_getinstrument
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_trade_getinstrument'
AS
SELECT
  InstrumentID,
  BuyCurrencyID,
  SellCurrencyID,
  InstrumentTypeID,
  Name,
  TradeRange,
  DollarRatio,
  Passport,
  PipDifferenceThreshold,
  IsMajor,
  Industry,
  ExchangeID
FROM main.trading.bronze_etoro_trade_getinstrument

In [0]:
%sql
-- Reg_Ext_Trade_InstrumentMetaData: full snapshot of Trade.InstrumentMetaData
-- SSIS parity: full table snapshot. This is the input for InstrumentMetaData_SpecialChar_Conversion.

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_instrumentmetadata
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_trade_instrumentmetadata'
AS
SELECT
  InstrumentID,
  InstrumentDisplayName,
  InstrumentTypeImage,
  Ticker,
  ChartTicker,
  InstrumentImageSmall,
  InstrumentImageMedium,
  InstrumentImageLarge,
  Exchange,
  Industry,
  CompanyInfo,
  DailyRolloverFee,
  WeekendRolloverFee,
  ContractRolloverFee,
  InstrumentVisible,
  Symbol,
  CandleTimeframeGroup,
  SymbolFull,
  Tradable,
  ExchangeID,
  StocksIndustryID,
  ISINCode,
  ISINCountryCode,
  ContractExpire,
  InstrumentTypeSubCategoryID,
  InstrumentTypeID,
  PriceSourceID,
  Cusip,
  CreateDate,
  UnderlyingExchangeID,
  DbLoginName,
  AppLoginName,
  SysStartTime,
  SysEndTime
FROM main.trading.bronze_etoro_trade_instrumentmetadata

In [0]:
%sql
-- Reg_Ext_DictionaryCurrency: Currency dictionary snapshot
-- SSIS parity: full table snapshot from Dictionary.Currency

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrency
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_dictionarycurrency'
AS
SELECT
  CurrencyID,
  CurrencyTypeID,
  Name,
  Abbreviation,
  Mask,
  EEAStockExchange,
  ISINCode,
  CurrencySymbol,
  InterestRateID
FROM main.general.bronze_etoro_dictionary_currency

In [0]:
%sql
-- Reg_Ext_DictionaryCurrencyType: CurrencyType dictionary snapshot
-- SSIS parity: full table snapshot from Dictionary.CurrencyType

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrencytype
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_dictionarycurrencytype'
AS
SELECT
  CurrencyTypeID,
  Name,
  MinPositionAmountAbsolute,
  Priority,
  PricesBy,
  SLTPApproachPercent,
  ImageUrl
FROM main.general.bronze_etoro_dictionary_currencytype

In [0]:
%sql
-- Reg_Ext_HedgeExecutionLog: Hedge execution log for report-date window
-- SSIS parity: ExecutionTime window filter + exclude (ProviderExecID IS NULL AND OrderState = 4)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgeexecutionlog
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_hedgeexecutionlog'
AS
WITH run_window AS (
  SELECT
    CAST('${report_date}' AS TIMESTAMP) AS window_start_ts,
    CAST(date_add(CAST('${report_date}' AS DATE), 1) AS TIMESTAMP) AS window_end_ts
)
SELECT
  src.LogTime,
  src.HedgeServerID,
  src.LiquidityAccountID,
  src.InstrumentID,
  src.OrderID,
  src.ParentOrderID,
  src.Units,
  src.IsBuy,
  src.OrderState,
  src.ProviderOrderID,
  src.SendTime,
  src.ProviderExecID,
  src.ExecutionTime,
  src.ExecutionRate,
  src.FailID,
  src.FailReason,
  src.Success,
  src.ProviderPartyIds,
  src.ReceivedTime,
  src.ProviderUnits,
  src.RateIDAtSent,
  src.EMSOrderID
FROM main.dealing.bronze_etoro_hedge_executionlog src
JOIN run_window w
  ON src.ExecutionTime >= w.window_start_ts
 AND src.ExecutionTime < w.window_end_ts
WHERE NOT (src.ProviderExecID IS NULL AND src.OrderState = 4)

In [0]:
%sql
-- Reg_Ext_HedgeHBCExecutionLog: HBC execution log for report-date window
-- SSIS parity: StartTime/EndTime window filter, IsSuccess filter

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcexecutionlog
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_hedgehbcexecutionlog'
AS
WITH run_window AS (
  SELECT
    CAST('${report_date}' AS TIMESTAMP) AS window_start_ts,
    CAST(date_add(CAST('${report_date}' AS DATE), 1) AS TIMESTAMP) AS window_end_ts
)
SELECT
  src.ExecutionID,
  src.HedgeServerID,
  src.LiquidityAccountID,
  src.InstrumentID,
  src.IsBuy,
  src.IsSuccess,
  src.RequestAmountInLots,
  src.ExecutionAmountInLots,
  src.ExecutionRate,
  src.StartTime,
  src.EndTime,
  src.FailReason,
  src.LPExecutionRate,
  src.MarketRateIDAtExecutionEnd,
  src.ShouldWaitForConfirm
FROM main.dealing.bronze_etoro_hedge_hbcexecutionlog src
JOIN run_window w
  ON src.EndTime >= w.window_start_ts
 AND src.EndTime < w.window_end_ts
WHERE src.IsSuccess = true

In [0]:
%sql
-- Reg_Ext_HedgeHBCOrderLog: HBC order log for report-date window
-- SSIS parity: date window filter (source uses etr_ymd partition; SSIS used CreatedDate)
-- Note: Source table has StartTime/EndTime but no CreatedDate. Using etr_ymd partition for date filter.

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcorderlog
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_ext_hedgehbcorderlog'
AS
SELECT
  src.*
FROM main.dealing.bronze_etoro_hedge_hbcorderlog src
WHERE src.etr_ymd = CAST('${report_date}' AS DATE)

In [0]:
%sql
-- Reg_Instruments_ext: Instruments from PRODUCTION SCD source (RegReportDB)
-- SSIS parity: full snapshot of Reg_Instruments_ext
-- Source: main.regtech.gold_regreportdb_prod_dbo_reg_instruments_scd (exact SP source mirror)
-- Contains IsMifid, IsMifidByFCA, IsFuture columns natively (no FuturesMetaData join needed)
--
-- PREVIOUSLY used main.regtech.gold_regtech_reg_instruments_scd which had:
--   - IsMifidByESMA (not IsMifid) — wrong column name
--   - IsMifid flags CLEARED in current versions (only 316 tagged vs 10,526 in production)
--   - No IsFuture column (required LEFT JOIN to bronze_etoro_trade_futuresmetadata)

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/reg_instruments_ext'
AS
SELECT src.*
FROM main.regtech.gold_regreportdb_prod_dbo_reg_instruments_scd src
WHERE src.ValidTo > CAST('${report_date}' AS DATE)
  AND src.ValidFrom <= '${report_date}'

In [0]:
%sql
-- Validation: row counts for all Module 4B staging tables
SELECT 'reg_ext_trade_getinstrument' AS table_name, COUNT(*) AS row_count FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_getinstrument
UNION ALL SELECT 'reg_ext_trade_instrumentmetadata', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_instrumentmetadata
UNION ALL SELECT 'reg_ext_dictionarycurrency', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrency
UNION ALL SELECT 'reg_ext_dictionarycurrencytype', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrencytype
UNION ALL SELECT 'reg_ext_hedgeexecutionlog', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgeexecutionlog
UNION ALL SELECT 'reg_ext_hedgehbcexecutionlog', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcexecutionlog
UNION ALL SELECT 'reg_ext_hedgehbcorderlog', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcorderlog
UNION ALL SELECT 'reg_instruments_ext', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext